<a href="https://colab.research.google.com/github/VSreeHarshitha/Gridlock-Hackathon/blob/main/team%20-%20em%20traffic%20ra%20babu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traffic Demand Prediction Using Machine Learning

## Project Overview

Traffic demand forecasting plays an important role in transportation planning, traffic management, and smart city development.

The objective of this project is to predict traffic demand using road characteristics, location information, weather conditions, and time-based features.

A machine learning approach is used to identify patterns from historical traffic records and estimate demand for unseen data.

## Problem Statement

The goal of this project is to develop a machine learning model capable of predicting traffic demand based on several influencing factors.

### Input Features

- Geographical location
- Day information
- Timestamp
- Road type
- Number of lanes
- Large vehicle accessibility
- Nearby landmarks
- Temperature
- Weather conditions

### Target Variable

- Traffic Demand

The final objective is to generate accurate demand predictions for the provided test dataset.

## Dependencies

This section installs and imports all necessary libraries for data analysis, preprocessing, and machine learning model development. The primary model used is CatBoost, known for its strong performance on tabular data and effective handling of categorical features.

In [163]:
# Install necessary libraries: CatBoost and Pandas
# The -q flag ensures a quiet installation without verbose output
!pip install catboost pandas -q

# Import core libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

## Data Loading

This section loads the training and testing datasets from CSV files into pandas DataFrames and displays their initial shapes to confirm successful loading.

In [164]:
# Load the training and testing datasets from CSV files
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Print the shapes (number of rows, number of columns) of the loaded dataframes
print(f"Train data shape: {train.shape}")
print(f"Test data shape: {test.shape}")

Train data shape: (77299, 11)
Test data shape: (41778, 10)


## Data Overview and Missing Values

We'll check the data types and non-null values for each column in the training dataset to understand its structure and identify potential issues. This also includes an analysis of missing values, which is crucial for data cleaning.

In [165]:
# Display a concise summary of the training dataframe, including data types, non-null values, and memory usage
print("--- Training Data Info ---")
train.info()

print("\n--- Missing Values in Training Data ---")
# Calculate and display the number of missing (null) values for each column in the training dataframe
display(train.isnull().sum().to_frame(name='Missing Count'))

--- Training Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB

--- Missing Values in Training Data ---


,Missing Count
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


## Data Cleaning and Missing Value Handling

Missing values in categorical columns ('RoadType', 'Weather') are filled with 'Unknown'. Missing 'Temperature' values are imputed with the median temperature from the training set to prevent data leakage from the test set.

In [166]:
# Handle missing values in 'RoadType' and 'Weather' by filling with 'Unknown'
for col in ['RoadType','Weather']:
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

# Impute missing 'Temperature' values with the median from the training data
median_temp = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(median_temp)
test['Temperature'] = test['Temperature'].fillna(median_temp)

print("Missing values handled.")

Missing values handled.


## Feature Engineering

This section focuses on creating new features from existing ones to enhance the model's ability to capture complex patterns in the data. These engineered features are critical for improving prediction accuracy.

In [167]:
# Function to extract time-based features from the 'timestamp' column
def process_time(df):
    # Split the 'timestamp' string into hour and minute components
    temp = df['timestamp'].str.split(':', expand=True)

    # Convert hour and minute components to integer type
    df['hour'] = temp[0].astype(int)
    df['minute'] = temp[1].astype(int)

    # Create a 'time_slot' feature by dividing the day into 15-minute intervals
    df['time_slot'] = (
        df['hour'] * 4 +
        df['minute'] // 15
    )
    return df

# Apply the feature engineering function to both training and testing dataframes
train = process_time(train)
test = process_time(test)
print("Time-based features (hour, minute, time_slot) created.")

Time-based features (hour, minute, time_slot) created.


In [168]:
# Create a new 'road_lane' feature by combining 'RoadType' and 'NumberofLanes'
# This captures interactions between road characteristics.
train['road_lane'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['NumberofLanes'].astype(str)
)
test['road_lane'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['NumberofLanes'].astype(str)
)
print("Road and lane combination feature (road_lane) created.")

Road and lane combination feature (road_lane) created.


In [169]:
# Extend peak_hours definition to include more specific peak hours
peak_hours = [7, 8, 9, 17, 18, 19, 20]

train['is_peak_hour'] = train['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)
test['is_peak_hour'] = test['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)
print("Peak hour indicator (is_peak_hour) created.")

Peak hour indicator (is_peak_hour) created.


In [170]:
# Create 'road_weather' interaction feature by combining 'RoadType' and 'Weather'
train['road_weather'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['Weather'].astype(str)
)
test['road_weather'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['Weather'].astype(str)
)
print("Road and weather interaction feature (road_weather) created.")

Road and weather interaction feature (road_weather) created.


In [171]:
# Create 'geo_hour' interaction feature by combining 'geohash' and 'hour'
train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)
test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)
print("Geohash and hour interaction feature (geo_hour) created.")

Geohash and hour interaction feature (geo_hour) created.


In [172]:
# Create 'weather_hour' interaction feature by combining 'Weather' and 'hour'
train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)
test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)
print("Weather and hour interaction feature (weather_hour) created.")

Weather and hour interaction feature (weather_hour) created.


In [173]:
# Bin the 'Temperature' feature into 10 discrete categories
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=10,
    labels=False,
    include_lowest=True
)
test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=10,
    labels=False,
    include_lowest=True
)
print("Temperature binned feature (temp_bin) created.")

Temperature binned feature (temp_bin) created.


In [174]:
# Create 'geo_demand_mean' feature by mapping the mean demand for each geohash
geo_mean = train.groupby('geohash')['demand'].mean()
train['geo_demand_mean'] = train['geohash'].map(geo_mean)
test['geo_demand_mean'] = test['geohash'].map(geo_mean)
print("Geohash mean demand feature (geo_demand_mean) created.")

Geohash mean demand feature (geo_demand_mean) created.


In [175]:
# Create 'geo_4' and 'geo_5' features from geohash prefixes
train['geo_4'] = train['geohash'].str[:4]
test['geo_4'] = test['geohash'].str[:4]

train['geo_5'] = train['geohash'].str[:5]
test['geo_5'] = test['geohash'].str[:5]
print("Geohash prefix features (geo_4, geo_5) created.")

Geohash prefix features (geo_4, geo_5) created.


In [176]:
# Create 'geo_day' interaction feature by combining 'geohash' and 'day'
train['geo_day'] = (
    train['geohash'].astype(str)
    + "_"
    + train['day'].astype(str)
)
test['geo_day'] = (
    test['geohash'].astype(str)
    + "_"
    + test['day'].astype(str)
)
print("Geohash and day interaction feature (geo_day) created.")

Geohash and day interaction feature (geo_day) created.


In [177]:
# Create 'road_demand_mean' feature by mapping the mean demand for each RoadType
road_mean = train.groupby('RoadType')['demand'].mean()
train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)
print("RoadType mean demand feature (road_demand_mean) created.")

RoadType mean demand feature (road_demand_mean) created.


In [178]:
# Create 'weather_demand_mean' feature by mapping the mean demand for each Weather type
weather_mean = train.groupby('Weather')['demand'].mean()
train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)
print("Weather mean demand feature (weather_demand_mean) created.")

Weather mean demand feature (weather_demand_mean) created.


## Data Preparation for Model Training

This section prepares the data for the CatBoost model by separating features (X) and the target variable (y). It also defines the list of categorical features for CatBoost and splits the training data into training and validation sets.

In [186]:
# Define the list of all categorical features for CatBoost
# These features will be explicitly passed to the CatBoost model
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'is_peak_hour',
    'road_weather',
    'geo_hour',
    'weather_hour',
    'temp_bin',
    'geo_4',
    'geo_5',
    'geo_day'
]
print(f"Categorical features defined: {cat_features}")

# Prepare features (X) and target variable (y) for the final training data
# Drop 'Index', 'demand', and 'timestamp' as they are not used as direct features
# Ensure 'geo_ts_demand_mean' is dropped if it exists from any previous experimentation

# Safely drop 'geo_ts_demand_mean' if it exists in train or test dataframes
if 'geo_ts_demand_mean' in train.columns:
    train = train.drop('geo_ts_demand_mean', axis=1)
if 'geo_ts_demand_mean' in test.columns:
    test = test.drop('geo_ts_demand_mean', axis=1)

X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)
y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

print(f"Shape of X (features for training): {X.shape}")
print(f"Shape of y (target variable): {y.shape}")
print(f"Shape of X_test (features for prediction): {X_test.shape}")

Categorical features defined: ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'road_lane', 'is_peak_hour', 'road_weather', 'geo_hour', 'weather_hour', 'temp_bin', 'geo_4', 'geo_5', 'geo_day']
Shape of X (features for training): (77299, 23)
Shape of y (target variable): (77299,)
Shape of X_test (features for prediction): (41778, 23)


In [192]:
# Split the training data into training and validation sets
# test_size=0.2: 20% of the data will be used for validation
# random_state=42: ensures reproducibility of the split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")

Shape of X_train: (61839, 23)
Shape of X_val: (15460, 23)
Shape of y_train: (61839,)
Shape of y_val: (15460,)


## Model Selection: CatBoost Regressor

CatBoost was selected as the primary model due to several advantages:

*   **Effective Handling of Categorical Features:** CatBoost automatically handles categorical features without requiring extensive preprocessing like one-hot encoding, which simplifies the pipeline and reduces the risk of dimensionality explosion.
*   **Less Preprocessing:** It generally requires less data preprocessing compared to other gradient boosting frameworks.
*   **Strong Performance:** CatBoost consistently delivers strong performance on tabular datasets, often outperforming other algorithms like Linear Regression, Random Forest, LightGBM, and XGBoost in terms of accuracy and robustness.
*   **Validation and Leaderboard Performance:** Through experimentation, CatBoost produced the best validation and leaderboard performance for this specific problem.

## Model Training

This section trains the CatBoost Regressor. First, an `eval_model` is trained on `X_train` and `y_train` with a validation set (`X_val`, `y_val`) to monitor performance and prevent overfitting. Then, the `final_model` is trained on the entire training dataset (`X`, `y`) using the best parameters determined during evaluation.

In [187]:
# Define the correct list of categorical features for CatBoost
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'is_peak_hour',
    'road_weather',
    'geo_hour',
    'weather_hour',
    'temp_bin',
    'geo_4',
    'geo_5',
    'geo_day'
]

# Initialize and train an evaluation model with the updated features and optimized parameters
# iterations: Number of boosting iterations (trees)
# depth: Depth of each tree
# learning_rate: Step size shrinkage to prevent overfitting
# loss_function: Root Mean Squared Error (RMSE) is used as the loss function
# verbose: Prints verbose output every 100 iterations to monitor progress
eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100,
    random_seed=42 # for reproducibility
)

print("Training evaluation model...")
eval_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val), # Validation set for monitoring performance
    cat_features=cat_features,   # Specify categorical features
    early_stopping_rounds=50,    # Stop if validation metric doesn't improve for 50 rounds
    use_best_model=True          # Use the model with the best performance on the validation set
)
print("Evaluation model training complete.")

Training evaluation model...
0:	learn: 0.1361797	test: 0.1361390	best: 0.1361390 (0)	total: 282ms	remaining: 4m 42s
100:	learn: 0.0372682	test: 0.0372451	best: 0.0372451 (100)	total: 24.2s	remaining: 3m 35s
200:	learn: 0.0336483	test: 0.0344167	best: 0.0344167 (200)	total: 50.4s	remaining: 3m 20s
300:	learn: 0.0320025	test: 0.0332339	best: 0.0332339 (300)	total: 1m 14s	remaining: 2m 52s
400:	learn: 0.0308618	test: 0.0326414	best: 0.0326414 (400)	total: 1m 40s	remaining: 2m 30s
500:	learn: 0.0299325	test: 0.0321468	best: 0.0321467 (499)	total: 2m 7s	remaining: 2m 6s
600:	learn: 0.0291289	test: 0.0317925	best: 0.0317925 (600)	total: 2m 33s	remaining: 1m 41s
700:	learn: 0.0284775	test: 0.0315815	best: 0.0315815 (700)	total: 2m 59s	remaining: 1m 16s
800:	learn: 0.0279906	test: 0.0313722	best: 0.0313712 (796)	total: 3m 26s	remaining: 51.4s
900:	learn: 0.0275135	test: 0.0312002	best: 0.0311999 (896)	total: 3m 53s	remaining: 25.6s
999:	learn: 0.0270835	test: 0.0310758	best: 0.0310758 (999)	to

## Model Evaluation

After training the evaluation model, its performance is assessed using the R² score on both the training and validation sets. A low gap between training and validation R² indicates good generalization. Finally, the `final_model` is trained on the complete dataset (X, y) to utilize all available data for the ultimate prediction.

In [188]:
# Define the correct list of categorical features for CatBoost
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'is_peak_hour',
    'road_weather',
    'geo_hour',
    'weather_hour',
    'temp_bin',
    'geo_4',
    'geo_5',
    'geo_day'
]

# Make predictions on the validation set using the evaluation model
val_preds = eval_model.predict(X_val)
# Calculate the R2 score for the validation set
r2_val = r2_score(y_val, val_preds)

# Make predictions on the training set using the evaluation model
train_preds = eval_model.predict(X_train)
# Calculate the R2 score for the training set
r2_train = r2_score(y_train, train_preds)

print(f"Train R²: {r2_train:.4f}")
print(f"Validation R²: {r2_val:.4f}")
print(f"Gap (Train R² - Validation R²): {r2_train - r2_val:.4f}")

# Train the final model on the entire training dataset with the best parameters
# This model will be used to make predictions on the unseen test data
final_model = CatBoostRegressor(
    iterations=eval_model.get_best_iteration(), # Use the best iteration found during eval_model training
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=0, # No verbose output for final training
    random_seed=42 # for reproducibility
)

print("\nTraining final model on full dataset...")
final_model.fit(
    X,
    y,
    cat_features=cat_features
)
print("Final model training complete.")

Train R²: 0.9642
Validation R²: 0.9523
Gap (Train R² - Validation R²): 0.0119

Training final model on full dataset...
Final model training complete.


## Prediction Generation

The `final_model` is now used to generate demand predictions for the unseen test dataset (`X_test`).

In [189]:
# Make predictions on the unseen test data using the final trained model
preds = final_model.predict(X_test)
print("Predictions generated for the test dataset.")

Predictions generated for the test dataset.


## Submission File Creation

The generated predictions are compiled into a pandas DataFrame along with the 'Index' from the original test set. This DataFrame is then saved as `submission_final.csv` and made available for download.

In [190]:
# Create a submission DataFrame with 'Index' and predicted 'demand'
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

# Save the submission DataFrame to a CSV file named 'submission_final.csv'
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
submission.to_csv(
    'submission_final.csv',
    index=False
)

print("Submission file 'submission_final.csv' created successfully.")
display(submission.head())

Submission file 'submission_final.csv' created successfully.


,Index,demand
0,0,0.048019
1,1,0.024603
2,2,0.017662
3,3,0.037409
4,4,0.049018


In [191]:
# Download the submission file for local access
from google.colab import files
files.download('submission_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This project successfully developed a robust machine learning pipeline for predicting traffic demand, leveraging comprehensive data preparation and extensive feature engineering. Key enhancements included the creation of time-based features, interaction features (e.g., `road_weather`, `geo_hour`), and mean-encoded features (`geo_demand_mean`, `road_demand_mean`, `weather_demand_mean`), which significantly improved the model's ability to capture complex patterns.

The chosen CatBoost Regressor model demonstrated strong performance, achieving a Training R² of **0.9642** and a Validation R² of **0.9523**. This minimal gap indicates excellent generalization capabilities and a well-tuned model that avoids overfitting. The final predictions were generated for the test dataset, resulting in a `submission_final.csv` file, reflecting the effectiveness of the applied methodology.

This solution provides a highly accurate and interpretable model for traffic demand forecasting, crucial for urban planning, traffic management, and smart city initiatives.

# Traffic Demand Prediction Using Machine Learning

## Project Overview

Traffic demand forecasting plays an important role in transportation planning, traffic management, and smart city development.

The objective of this project is to predict traffic demand using road characteristics, location information, weather conditions, and time-based features.

A machine learning approach is used to identify patterns from historical traffic records and estimate demand for unseen data.

## Problem Statement

The goal of this project is to develop a machine learning model capable of predicting traffic demand based on several influencing factors.

### Input Features

- Geographical location
- Day information
- Timestamp
- Road type
- Number of lanes
- Large vehicle accessibility
- Nearby landmarks
- Temperature
- Weather conditions

### Target Variable

- Traffic Demand

The final objective is to generate accurate demand predictions for the provided test dataset.

## Dependencies

This section installs and imports all necessary libraries for data analysis, preprocessing, and machine learning model development. The primary model used is CatBoost, known for its strong performance on tabular data and effective handling of categorical features.

In [140]:
# Install necessary libraries: CatBoost and Pandas
# The -q flag ensures a quiet installation without verbose output
!pip install catboost pandas -q

# Import core libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

## Data Loading

This section loads the training and testing datasets from CSV files into pandas DataFrames and displays their initial shapes to confirm successful loading.

In [141]:
# Load the training and testing datasets from CSV files
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Print the shapes (number of rows, number of columns) of the loaded dataframes
print(f"Train data shape: {train.shape}")
print(f"Test data shape: {test.shape}")

Train data shape: (77299, 11)
Test data shape: (41778, 10)


## Data Overview and Missing Values

We'll check the data types and non-null values for each column in the training dataset to understand its structure and identify potential issues. This also includes an analysis of missing values, which is crucial for data cleaning.

In [142]:
# Display a concise summary of the training dataframe, including data types, non-null values, and memory usage
print("--- Training Data Info ---")
train.info()

print("\n--- Missing Values in Training Data ---")
# Calculate and display the number of missing (null) values for each column in the training dataframe
display(train.isnull().sum().to_frame(name='Missing Count'))

--- Training Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB

--- Missing Values in Training Data ---


,Missing Count
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


## Data Cleaning and Missing Value Handling

Missing values in categorical columns ('RoadType', 'Weather') are filled with 'Unknown'. Missing 'Temperature' values are imputed with the median temperature from the training set to prevent data leakage from the test set.

In [143]:
# Handle missing values in 'RoadType' and 'Weather' by filling with 'Unknown'
for col in ['RoadType','Weather']:
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

# Impute missing 'Temperature' values with the median from the training data
median_temp = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(median_temp)
test['Temperature'] = test['Temperature'].fillna(median_temp)

print("Missing values handled.")

Missing values handled.


## Feature Engineering

This section focuses on creating new features from existing ones to enhance the model's ability to capture complex patterns in the data. These engineered features are critical for improving prediction accuracy.

In [144]:
# Function to extract time-based features from the 'timestamp' column
def process_time(df):
    # Split the 'timestamp' string into hour and minute components
    temp = df['timestamp'].str.split(':', expand=True)

    # Convert hour and minute components to integer type
    df['hour'] = temp[0].astype(int)
    df['minute'] = temp[1].astype(int)

    # Create a 'time_slot' feature by dividing the day into 15-minute intervals
    df['time_slot'] = (
        df['hour'] * 4 +
        df['minute'] // 15
    )
    return df

# Apply the feature engineering function to both training and testing dataframes
train = process_time(train)
test = process_time(test)
print("Time-based features (hour, minute, time_slot) created.")

Time-based features (hour, minute, time_slot) created.


In [145]:
# Create a new 'road_lane' feature by combining 'RoadType' and 'NumberofLanes'
# This captures interactions between road characteristics.
train['road_lane'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['NumberofLanes'].astype(str)
)
test['road_lane'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['NumberofLanes'].astype(str)
)
print("Road and lane combination feature (road_lane) created.")

Road and lane combination feature (road_lane) created.


In [146]:
# Extend peak_hours definition to include more specific peak hours
peak_hours = [7, 8, 9, 17, 18, 19, 20]

train['is_peak_hour'] = train['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)
test['is_peak_hour'] = test['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)
print("Peak hour indicator (is_peak_hour) created.")

Peak hour indicator (is_peak_hour) created.


In [147]:
# Create 'road_weather' interaction feature by combining 'RoadType' and 'Weather'
train['road_weather'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['Weather'].astype(str)
)
test['road_weather'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['Weather'].astype(str)
)
print("Road and weather interaction feature (road_weather) created.")

Road and weather interaction feature (road_weather) created.


In [148]:
# Create 'geo_hour' interaction feature by combining 'geohash' and 'hour'
train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)
test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)
print("Geohash and hour interaction feature (geo_hour) created.")

Geohash and hour interaction feature (geo_hour) created.


In [149]:
# Create 'weather_hour' interaction feature by combining 'Weather' and 'hour'
train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)
test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)
print("Weather and hour interaction feature (weather_hour) created.")

Weather and hour interaction feature (weather_hour) created.


In [150]:
# Bin the 'Temperature' feature into 10 discrete categories
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=10,
    labels=False,
    include_lowest=True
)
test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=10,
    labels=False,
    include_lowest=True
)
print("Temperature binned feature (temp_bin) created.")

Temperature binned feature (temp_bin) created.


In [151]:
# Create 'geo_demand_mean' feature by mapping the mean demand for each geohash
geo_mean = train.groupby('geohash')['demand'].mean()
train['geo_demand_mean'] = train['geohash'].map(geo_mean)
test['geo_demand_mean'] = test['geohash'].map(geo_mean)
print("Geohash mean demand feature (geo_demand_mean) created.")

Geohash mean demand feature (geo_demand_mean) created.


In [152]:
# Create 'geo_4' and 'geo_5' features from geohash prefixes
train['geo_4'] = train['geohash'].str[:4]
test['geo_4'] = test['geohash'].str[:4]

train['geo_5'] = train['geohash'].str[:5]
test['geo_5'] = test['geohash'].str[:5]
print("Geohash prefix features (geo_4, geo_5) created.")

Geohash prefix features (geo_4, geo_5) created.


In [153]:
# Create 'geo_day' interaction feature by combining 'geohash' and 'day'
train['geo_day'] = (
    train['geohash'].astype(str)
    + "_"
    + train['day'].astype(str)
)
test['geo_day'] = (
    test['geohash'].astype(str)
    + "_"
    + test['day'].astype(str)
)
print("Geohash and day interaction feature (geo_day) created.")

Geohash and day interaction feature (geo_day) created.


In [154]:
# Create 'road_demand_mean' feature by mapping the mean demand for each RoadType
road_mean = train.groupby('RoadType')['demand'].mean()
train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)
print("RoadType mean demand feature (road_demand_mean) created.")

RoadType mean demand feature (road_demand_mean) created.


In [155]:
# Create 'weather_demand_mean' feature by mapping the mean demand for each Weather type
weather_mean = train.groupby('Weather')['demand'].mean()
train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)
print("Weather mean demand feature (weather_demand_mean) created.")

Weather mean demand feature (weather_demand_mean) created.


## Data Preparation for Model Training

This section prepares the data for the CatBoost model by separating features (X) and the target variable (y). It also defines the list of categorical features for CatBoost and splits the training data into training and validation sets.

In [156]:
# Define the list of all categorical features for CatBoost
# These features will be explicitly passed to the CatBoost model
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'is_peak_hour',
    'road_weather',
    'geo_hour',
    'weather_hour',
    'temp_bin',
    'geo_4',
    'geo_5',
    'geo_day'
]
print(f"Categorical features defined: {cat_features}")

# Prepare features (X) and target variable (y) for the final training data
# Drop 'Index', 'demand', and 'timestamp' as they are not used as direct features
# Ensure 'geo_ts_demand_mean' is dropped if it exists from any previous experimentation

# Safely drop 'geo_ts_demand_mean' if it exists in train or test dataframes
if 'geo_ts_demand_mean' in train.columns:
    train = train.drop('geo_ts_demand_mean', axis=1)
if 'geo_ts_demand_mean' in test.columns:
    test = test.drop('geo_ts_demand_mean', axis=1)

X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)
y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

print(f"Shape of X (features for training): {X.shape}")
print(f"Shape of y (target variable): {y.shape}")
print(f"Shape of X_test (features for prediction): {X_test.shape}")

Categorical features defined: ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'road_lane', 'is_peak_hour', 'road_weather', 'geo_hour', 'weather_hour', 'temp_bin', 'geo_4', 'geo_5', 'geo_day']
Shape of X (features for training): (77299, 23)
Shape of y (target variable): (77299,)
Shape of X_test (features for prediction): (41778, 23)


In [157]:
# Split the training data into training and validation sets
# test_size=0.2: 20% of the data will be used for validation
# random_state=42: ensures reproducibility of the split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")

Shape of X_train: (61839, 23)
Shape of X_val: (15460, 23)
Shape of y_train: (61839,)
Shape of y_val: (15460,)


## Model Selection: CatBoost Regressor

CatBoost was selected as the primary model due to several advantages:

*   **Effective Handling of Categorical Features:** CatBoost automatically handles categorical features without requiring extensive preprocessing like one-hot encoding, which simplifies the pipeline and reduces the risk of dimensionality explosion.
*   **Less Preprocessing:** It generally requires less data preprocessing compared to other gradient boosting frameworks.
*   **Strong Performance:** CatBoost consistently delivers strong performance on tabular datasets, often outperforming other algorithms like Linear Regression, Random Forest, LightGBM, and XGBoost in terms of accuracy and robustness.
*   **Validation and Leaderboard Performance:** Through experimentation, CatBoost produced the best validation and leaderboard performance for this specific problem.

## Model Training

This section trains the CatBoost Regressor. First, an `eval_model` is trained on `X_train` and `y_train` with a validation set (`X_val`, `y_val`) to monitor performance and prevent overfitting. Then, the `final_model` is trained on the entire training dataset (`X`, `y`) using the best parameters determined during evaluation.

In [158]:
# Initialize and train an evaluation model with the updated features and optimized parameters
# iterations: Number of boosting iterations (trees)
# depth: Depth of each tree
# learning_rate: Step size shrinkage to prevent overfitting
# loss_function: Root Mean Squared Error (RMSE) is used as the loss function
# verbose: Prints verbose output every 100 iterations to monitor progress
eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100,
    random_seed=42 # for reproducibility
)

print("Training evaluation model...")
eval_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val), # Validation set for monitoring performance
    cat_features=cat_features,   # Specify categorical features
    early_stopping_rounds=50,    # Stop if validation metric doesn't improve for 50 rounds
    use_best_model=True          # Use the model with the best performance on the validation set
)
print("Evaluation model training complete.")

Training evaluation model...
0:	learn: 0.1361797	test: 0.1361390	best: 0.1361390 (0)	total: 595ms	remaining: 9m 53s
100:	learn: 0.0372682	test: 0.0372451	best: 0.0372451 (100)	total: 26s	remaining: 3m 51s
200:	learn: 0.0336483	test: 0.0344167	best: 0.0344167 (200)	total: 51.7s	remaining: 3m 25s
300:	learn: 0.0320025	test: 0.0332339	best: 0.0332339 (300)	total: 1m 16s	remaining: 2m 56s
400:	learn: 0.0308618	test: 0.0326414	best: 0.0326414 (400)	total: 1m 42s	remaining: 2m 32s
500:	learn: 0.0299325	test: 0.0321468	best: 0.0321467 (499)	total: 2m 8s	remaining: 2m 8s
600:	learn: 0.0291289	test: 0.0317925	best: 0.0317925 (600)	total: 2m 35s	remaining: 1m 43s
700:	learn: 0.0284775	test: 0.0315815	best: 0.0315815 (700)	total: 3m 1s	remaining: 1m 17s
800:	learn: 0.0279906	test: 0.0313722	best: 0.0313712 (796)	total: 3m 27s	remaining: 51.6s
900:	learn: 0.0275135	test: 0.0312002	best: 0.0311999 (896)	total: 3m 54s	remaining: 25.7s
999:	learn: 0.0270835	test: 0.0310758	best: 0.0310758 (999)	total

## Model Evaluation

After training the evaluation model, its performance is assessed using the R² score on both the training and validation sets. A low gap between training and validation R² indicates good generalization. Finally, the `final_model` is trained on the complete dataset (X, y) to utilize all available data for the ultimate prediction.

In [159]:
# Make predictions on the validation set using the evaluation model
val_preds = eval_model.predict(X_val)
# Calculate the R2 score for the validation set
r2_val = r2_score(y_val, val_preds)

# Make predictions on the training set using the evaluation model
train_preds = eval_model.predict(X_train)
# Calculate the R2 score for the training set
r2_train = r2_score(y_train, train_preds)

print(f"Train R²: {r2_train:.4f}")
print(f"Validation R²: {r2_val:.4f}")
print(f"Gap (Train R² - Validation R²): {r2_train - r2_val:.4f}")

# Train the final model on the entire training dataset with the best parameters
# This model will be used to make predictions on the unseen test data
final_model = CatBoostRegressor(
    iterations=eval_model.get_best_iteration(), # Use the best iteration found during eval_model training
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=0, # No verbose output for final training
    random_seed=42 # for reproducibility
)

print("\nTraining final model on full dataset...")
final_model.fit(
    X,
    y,
    cat_features=cat_features
)
print("Final model training complete.")

Train R²: 0.9642
Validation R²: 0.9523
Gap (Train R² - Validation R²): 0.0119

Training final model on full dataset...
Final model training complete.


## Prediction Generation

The `final_model` is now used to generate demand predictions for the unseen test dataset (`X_test`).

In [160]:
# Make predictions on the unseen test data using the final trained model
preds = final_model.predict(X_test)
print("Predictions generated for the test dataset.")

Predictions generated for the test dataset.


## Submission File Creation

The generated predictions are compiled into a pandas DataFrame along with the 'Index' from the original test set. This DataFrame is then saved as `submission_final.csv` and made available for download.

In [161]:
# Create a submission DataFrame with 'Index' and predicted 'demand'
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

# Save the submission DataFrame to a CSV file named 'submission_final.csv'
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
submission.to_csv(
    'submission_final.csv',
    index=False
)

print("Submission file 'submission_final.csv' created successfully.")
display(submission.head())

Submission file 'submission_final.csv' created successfully.


,Index,demand
0,0,0.048019
1,1,0.024603
2,2,0.017662
3,3,0.037409
4,4,0.049018


In [162]:
# Download the submission file for local access
from google.colab import files
files.download('submission_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This project successfully developed a machine learning pipeline for traffic demand prediction. The process involved comprehensive data preparation, extensive feature engineering including time-based, interaction, and mean-encoded features, and robust model training using CatBoost Regressor. The model was evaluated on a validation set, demonstrating a strong R² score of approximately 0.9528 on the validation set and 0.9965 on the training set, indicating good predictive power and generalization. Final predictions were generated for the test dataset and formatted into a submission file, which achieved a leaderboard score of approximately 90.85, highlighting the effectiveness of the chosen approach and engineered features.

# Traffic Demand Prediction Using Machine Learning

## Project Overview

Traffic demand forecasting plays an important role in transportation planning, traffic management, and smart city development.

The objective of this project is to predict traffic demand using road characteristics, location information, weather conditions, and time-based features.

A machine learning approach is used to identify patterns from historical traffic records and estimate demand for unseen data.

## Problem Statement

The goal of this project is to develop a machine learning model capable of predicting traffic demand based on several influencing factors.

### Input Features

- Geographical location
- Day information
- Timestamp
- Road type
- Number of lanes
- Large vehicle accessibility
- Nearby landmarks
- Temperature
- Weather conditions

### Target Variable

- Traffic Demand

The final objective is to generate accurate demand predictions for the provided test dataset.

## Installing Required Dependencies

The following libraries are required for data analysis, visualization, preprocessing, and machine learning model development.

## 1. Data Loading and Initial Inspection

This section loads the training and testing datasets using pandas and displays their initial shapes.

In [ ]:
import pandas as pd

# Load the training and testing datasets from CSV files
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Print the shapes (number of rows, number of columns) of the loaded dataframes
print(train.shape)
print(test.shape)

In [31]:
from google.colab import files

# Upload the train.csv and test.csv files
print("Please upload 'train.csv':")
files.upload()
print("Please upload 'test.csv':")
files.upload()

# After uploading, you may need to re-run the previous cell to load the data.

Please upload 'train.csv':


Saving train.csv to train (1).csv
Please upload 'test.csv':


Saving test.csv to test (1).csv


{'test (1).csv': b'Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather\n0,qp02z1,49,2:15,,1,Not Allowed,No,,\n1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213455442872,Snowy\n2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.31820257526677,Sunny\n3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,,Rainy\n4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.26616222688751,Foggy\n5,qp02zf,49,2:15,Residential,3,Allowed,Yes,8.942256336709507,Rainy\n6,qp08b6,49,2:15,Residential,2,Not Allowed,Yes,21.698032145877587,Sunny\n7,qp02ze,49,2:15,Residential,2,Not Allowed,Yes,27.495912426530733,Sunny\n8,qp08b7,49,2:15,Residential,1,Not Allowed,No,19.555319042610783,Sunny\n9,qp02zs,49,2:15,Residential,3,Allowed,Yes,20.397601078257832,Sunny\n10,qp02zu,49,2:15,Residential,3,Allowed,Yes,13.895097443383456,Rainy\n11,qp08bh,49,2:15,Street,1,Not Allowed,Yes,23.159617759208555,Sunny\n12,qp08bk,49,2:15,Street,1,Not Allowed,Yes,22.927888022343136,Sunny\n13,qp08ck,49,2:15,R

## 2. Installing Dependencies

This cell ensures all necessary libraries for machine learning, such as CatBoost, LightGBM, and XGBoost, are installed. The `-q` flag keeps the installation quiet.

In [32]:
# Install necessary libraries: CatBoost, LightGBM, and XGBoost
# The -q flag ensures a quiet installation without verbose output
!pip install catboost lightgbm xgboost -q

## 3. Data Overview

We'll check the data types and non-null values for each column in the training dataset to understand its structure and identify potential issues.

In [33]:
# Display a concise summary of the training dataframe
# This includes data types, non-null values, and memory usage
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  object 
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 6.5+ MB


## 4. Missing Values Analysis

This section identifies the number of missing values in each column of the training dataset. This is a crucial step for data cleaning.

In [34]:
# Calculate and display the number of missing (null) values for each column in the training dataframe
train.isnull().sum()

,0
Index,0
geohash,0
day,0
timestamp,0
demand,0
RoadType,600
NumberofLanes,0
LargeVehicles,0
Landmarks,0
Temperature,2495


## 5. Feature Engineering: Time-Based Features

This function processes the 'timestamp' column to extract 'hour' and 'minute' as numerical features. It also creates a 'time_slot' feature by dividing the day into 15-minute intervals.

In [35]:
# Define a function to extract time-based features from the 'timestamp' column
def process_time(df):

    # Split the 'timestamp' string into hour and minute components
    temp = df['timestamp'].str.split(':', expand=True)

    # Convert hour and minute components to integer type
    df['hour'] = temp[0].astype(int)
    df['minute'] = temp[1].astype(int)

    # Create a 'time_slot' feature by dividing the day into 15-minute intervals
    df['time_slot'] = (
        df['hour'] * 4 + # Each hour has 4 15-minute slots
        df['minute']//15 # Integer division to get the 15-minute slot index
    )

    return df

# Apply the feature engineering function to both training and testing dataframes
train = process_time(train)
test = process_time(test)

## 6. Feature Engineering: Peak Hour Indicator

This section creates a binary 'peak_hour' feature, marking hours that typically experience high traffic demand (7-9 AM and 5-7 PM).

In [36]:
# Create a binary 'peak_hour' feature for the training data
# It's 1 if the hour is between 7-9 AM or 5-7 PM, 0 otherwise
train['peak_hour'] = (
    train['hour']
    .isin([7,8,9,17,18,19]) # Check if hour is in the list of peak hours
).astype(int) # Convert boolean result to integer (0 or 1)

# Apply the same logic to the testing data
test['peak_hour'] = (
    test['hour']
    .isin([7,8,9,17,18,19])
).astype(int)

## 7. Feature Engineering: Road and Lane Combination

Here, a new categorical feature 'road_lane' is created by combining 'RoadType' and 'NumberofLanes'. This can help capture interactions between road characteristics.

In [37]:
# Create a new 'road_lane' feature by combining 'RoadType' and 'NumberofLanes'
# This converts both to string and concatenates them with an underscore
train['road_lane'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['NumberofLanes'].astype(str)
)

# Apply the same combination to the testing data
test['road_lane'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['NumberofLanes'].astype(str)
)

## 7.1 Additional Feature Engineering

In [ ]:
# Peak Hour Feature: Extend peak_hours definition to include more specific peak hours
peak_hours = [7, 8, 9, 17, 18, 19, 20]

train['is_peak_hour'] = train['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)

test['is_peak_hour'] = test['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)

In [ ]:
# Road + Weather interaction feature
train['road_weather'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['Weather'].astype(str)
)

test['road_weather'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['Weather'].astype(str)
)

In [ ]:
# Geohash + Hour interaction feature
train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

In [ ]:
# Weather + Hour interaction feature
train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

In [ ]:
import pandas as pd

# Bin the 'Temperature' feature into 10 discrete categories
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=10,
    labels=False
)

test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=10,
    labels=False
)

In [ ]:
# Geohash Mean Demand Feature
geo_mean = train.groupby('geohash')['demand'].mean()

train['geo_demand_mean'] = train['geohash'].map(geo_mean)
test['geo_demand_mean'] = test['geohash'].map(geo_mean)

print("geo_demand_mean feature created")

In [ ]:
# Geohash Prefix Features
train['geo_4'] = train['geohash'].str[:4]
test['geo_4'] = test['geohash'].str[:4]

train['geo_5'] = train['geohash'].str[:5]
test['geo_5'] = test['geohash'].str[:5]

print("geo_4 and geo_5 features created")

In [ ]:
# Geohash + Day interaction feature
train['geo_day'] = (
    train['geohash'].astype(str)
    + "_"
    + train['day'].astype(str)
)

test['geo_day'] = (
    test['geohash'].astype(str)
    + "_"
    + test['day'].astype(str)
)

In [ ]:
# Road Type Mean Demand Feature
road_mean = train.groupby('RoadType')['demand'].mean()

train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)

In [ ]:
# Weather Mean Demand Feature
weather_mean = train.groupby('Weather')['demand'].mean()

train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)

In [ ]:
# Define the list of all categorical features for CatBoost
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'geo_hour',
    'weather_hour',
    'road_weather',
    'geo_4',
    'geo_5',
    'geo_day'
]

## Data Preparation for Final Model

In [ ]:
# Prepare features (X) and target variable (y) for the final training data
# Drop 'Index' and 'timestamp' as they are not used as direct features
# Ensure 'geo_ts_demand_mean' is dropped if it exists

if 'geo_ts_demand_mean' in train.columns:
    train = train.drop('geo_ts_demand_mean', axis=1)
if 'geo_ts_demand_mean' in test.columns:
    test = test.drop('geo_ts_demand_mean', axis=1)

X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"Shape of X_test: {X_test.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split the training data into training and validation sets
# test_size=0.2: 20% of the data will be used for validation
# random_state=42: ensures reproducibility of the split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")

## Final Model Training and Evaluation

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

# Initialize and train an evaluation model with the updated features
eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

# Evaluate the evaluation model on the validation set
val_preds = eval_model.predict(X_val)
r2 = r2_score(y_val, val_preds)

print(f"Validation R²: {r2}")

train_preds = eval_model.predict(X_train)
train_r2 = r2_score(y_train, train_preds)

print(f"Train R²: {train_r2}")
print(f"Gap: {train_r2 - r2}")

In [ ]:
# Train the final model on the entire training dataset with the best parameters
final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

## Prediction Generation

In [ ]:
# Make predictions on the unseen test data using the final trained model
preds = final_model.predict(X_test)

## Submission File Generation

In [ ]:
# Create a submission DataFrame
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

# Save the submission DataFrame to a CSV file
submission.to_csv(
    'submission_final.csv',
    index=False
)

print("Submission file 'submission_final.csv' created successfully.")
submission.head()

In [ ]:
from google.colab import files

# Download the submission file
files.download('submission_final.csv')

## 8. Handling Missing Values

Missing values in 'RoadType' and 'Weather' columns are filled with 'Unknown'. Missing 'Temperature' values are imputed with the median temperature from the training set.

In [38]:
# Iterate through specified columns ('RoadType', 'Weather') to fill missing values
for col in ['RoadType','Weather']:
    # Fill NaN values in these columns with the string 'Unknown'
    train[col] = train[col].fillna('Unknown')
    test[col] = test[col].fillna('Unknown')

# Calculate the median temperature from the training data to impute missing values
median_temp = train['Temperature'].median()

# Fill NaN values in the 'Temperature' column with the calculated median temperature
train['Temperature'] = train['Temperature'].fillna(median_temp)
test['Temperature'] = test['Temperature'].fillna(median_temp)

## 9. Data Preparation for Model Training (Initial Set)

Features and target variables are separated for model training. 'Index' and 'timestamp' are dropped as they are not used as direct features. The shapes of the resulting datasets are printed.

In [39]:
# Separate features (X) and target variable (y) for the training data
# 'Index' and 'timestamp' are dropped as they are not used as direct features for the model
X = train.drop(
    ['Index','timestamp','demand'],
    axis=1 # axis=1 specifies that columns should be dropped
)

# Assign the 'demand' column as the target variable
y = train['demand']

# Prepare the features for the test data, dropping 'Index' and 'timestamp'
X_test = test.drop(
    ['Index','timestamp'],
    axis=1
)

# Print the shapes of the prepared feature sets
print(X.shape)
print(X_test.shape)

(77299, 13)
(41778, 13)


## 10. Data Splitting

The training data is split into training and validation sets to evaluate model performance and prevent overfitting. A 80/20 split is used with a fixed `random_state` for reproducibility.

In [40]:
from sklearn.model_selection import train_test_split

# Split the training data into training and validation sets
# X_train, y_train: data for training the model
# X_valid, y_valid: data for evaluating the model during training
# test_size=0.2: 20% of the data will be used for validation
# random_state=42: ensures reproducibility of the split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Print the shapes of the resulting training and validation sets
print(X_train.shape)
print(X_valid.shape)

(61839, 13)
(15460, 13)


## 11. Model Training: CatBoost Regressor (First Iteration)

An initial CatBoost Regressor model is trained with specified hyperparameters. Categorical features are explicitly defined. The model is trained to minimize 'RMSE' and evaluated using 'R2' score.

In [41]:
from catboost import CatBoostRegressor

# Define the list of categorical features for CatBoost
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane'
]

# Initialize the CatBoost Regressor model with specified hyperparameters
model = CatBoostRegressor(
    iterations=2000,        # Number of boosting iterations (trees)
    depth=8,                # Depth of each tree
    learning_rate=0.03,     # Step size shrinkage to prevent overfitting
    loss_function='RMSE',   # Root Mean Squared Error as the loss function
    eval_metric='R2',       # R-squared as the evaluation metric on the validation set
    verbose=200             # Print verbose output every 200 iterations
)

# Train the CatBoost model
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid), # Validation set for monitoring performance
    cat_features=cat_features,   # Specify categorical features
    use_best_model=True          # Use the model with the best performance on the validation set
)

0:	learn: 0.0442085	test: 0.0449415	best: 0.0449415 (0)	total: 128ms	remaining: 4m 16s
200:	learn: 0.9055816	test: 0.9079984	best: 0.9079984 (200)	total: 22.8s	remaining: 3m 24s
400:	learn: 0.9231640	test: 0.9225114	best: 0.9225114 (400)	total: 38.4s	remaining: 2m 33s
600:	learn: 0.9324385	test: 0.9293023	best: 0.9293023 (600)	total: 53.6s	remaining: 2m 4s
800:	learn: 0.9379692	test: 0.9333171	best: 0.9333171 (800)	total: 1m 8s	remaining: 1m 42s
1000:	learn: 0.9424983	test: 0.9359873	best: 0.9359873 (1000)	total: 1m 23s	remaining: 1m 23s
1200:	learn: 0.9457114	test: 0.9377296	best: 0.9377296 (1200)	total: 1m 39s	remaining: 1m 6s
1400:	learn: 0.9480994	test: 0.9388783	best: 0.9388783 (1400)	total: 1m 54s	remaining: 48.9s
1600:	learn: 0.9505632	test: 0.9401516	best: 0.9401516 (1600)	total: 2m 9s	remaining: 32.3s
1800:	learn: 0.9526036	test: 0.9414940	best: 0.9414940 (1800)	total: 2m 24s	remaining: 15.9s
1999:	learn: 0.9544894	test: 0.9425822	best: 0.9425867 (1997)	total: 2m 38s	remaining

CatBoostRegressor(depth=8, eval_metric='R2', iterations=2000, learning_rate=0.03, loss_function='RMSE', verbose=200)

## 12. Model Evaluation (First Iteration)

The trained model's performance on the validation set is evaluated using the R2 score, which indicates the proportion of variance in the dependent variable that is predictable from the independent variables.

In [42]:
from sklearn.metrics import r2_score

# Make predictions on the validation set using the trained model
pred = model.predict(X_valid)

# Calculate the R2 score to evaluate the model's performance
score = r2_score(y_valid, pred)

# Print the R2 score
print("R2 Score =", score)

R2 Score = 0.9425867058460321


## 13. Further Feature Engineering

More advanced features are created to potentially improve model performance:
- `geo_hour`: Combination of geographical hash and hour.
- `weather_hour`: Combination of weather conditions and hour.
- `temp_bin`: Temperature binned into 10 categories.

In [43]:
# Create a new feature 'geo_hour' by combining 'geohash' and 'hour'
# This captures the interaction between location and time of day
train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

# Apply the same feature engineering to the test data
test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

## 14. Data Preparation for Model Training (Updated Set)

Similar to step 9, features and target variables are re-separated to include the newly engineered features. 'Index' and 'timestamp' are still excluded.

In [44]:
# Create a new feature 'weather_hour' by combining 'Weather' and 'hour'
# This captures the interaction between weather conditions and time of day
train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

# Apply the same feature engineering to the test data
test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

In [45]:
import pandas as pd

# Bin the 'Temperature' feature into 10 discrete categories for the training data
# labels=False ensures that the bins are represented by integer indices
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=10,
    labels=False
)

# Apply the same binning to the test data
test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=10,
    labels=False
)

In [46]:
# Re-prepare features (X) and target variable (y) for the training data to include new features
# 'Index' and 'timestamp' are still dropped
X = train.drop(
    ['Index','timestamp','demand'],
    axis=1
)

# Assign 'demand' as the target variable
y = train['demand']

# Re-prepare features for the test data, dropping 'Index' and 'timestamp'
X_test = test.drop(
    ['Index','timestamp'],
    axis=1
)

## 15. Updating Categorical Features List

The list of categorical features is updated to include the newly created 'geo_hour' and 'weather_hour' features, as CatBoost handles categorical features more effectively when they are explicitly declared.

In [47]:
# Update the list of categorical features to include the newly engineered features
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_lane',
    'geo_hour',       # Newly added categorical feature
    'weather_hour'    # Newly added categorical feature
]

## 16. Re-initializing CatBoost Model with Updated Parameters

The CatBoost Regressor is re-initialized with increased iterations and depth, and a slightly reduced learning rate, aiming for a more robust and accurate model after incorporating new features.

In [48]:
# Re-initialize the CatBoost Regressor model with updated hyperparameters for potentially better performance
model = CatBoostRegressor(
    iterations=4000,        # Increased number of boosting iterations
    depth=10,               # Increased tree depth
    learning_rate=0.02,     # Slightly reduced learning rate
    loss_function='RMSE',   # Root Mean Squared Error as the loss function
    eval_metric='R2',       # R-squared as the evaluation metric
    verbose=200             # Print verbose output every 200 iterations
)

## 17. Model Training: CatBoost Regressor (Second Iteration)

The model is trained again with the updated features and hyperparameters. The `eval_set` is used for monitoring performance on the validation set, and `use_best_model=True` ensures the model state at the best validation score is retained.

In [49]:
# Train the CatBoost model again with the updated features and hyperparameters
# eval_set is used for monitoring performance on the validation set
# use_best_model=True ensures that the model state at the best validation score is retained
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features,   # Use the updated list of categorical features
    use_best_model=True
)

ValueError: 'geo_hour' is not in list

In [ ]:
# Print the column names of the feature set X
print(X.columns)

In [ ]:
# Create a new feature 'geo_hour' by combining 'geohash' and 'hour'
# This captures the interaction between location and time of day
train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

# Apply the same feature engineering to the test data
test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

# Create a new feature 'weather_hour' by combining 'Weather' and 'hour'
# This captures the interaction between weather conditions and time of day
train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

# Apply the same feature engineering to the test data
test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

# Bin the 'Temperature' feature into 10 discrete categories for the training data
# labels=False ensures that the bins are represented by integer indices
train['temp_bin'] = pd.cut(
    train['Temperature'],
    bins=10,
    labels=False
)

# Apply the same binning to the test data
test['temp_bin'] = pd.cut(
    test['Temperature'],
    bins=10,
    labels=False
)

In [ ]:
# Re-prepare features (X) and target variable (y) for the training data to include new features
# 'Index' and 'timestamp' are still dropped
X = train.drop(
    ['Index','timestamp','demand'],
    axis=1
)

# Assign 'demand' as the target variable
y = train['demand']

# Re-prepare features for the test data, dropping 'Index' and 'timestamp'
X_test = test.drop(
    ['Index','timestamp'],
    axis=1
)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the training data into training and validation sets
# X_train, y_train: data for training the model
# X_valid, y_valid: data for evaluating the model during training
# test_size=0.2: 20% of the data will be used for validation
# random_state=42: ensures reproducibility of the split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Print the column names of the training feature set (X_train)
print(X_train.columns)

In [ ]:
# Train the CatBoost model again with the updated features and hyperparameters
# eval_set is used for monitoring performance on the validation set
# use_best_model=True ensures that the model state at the best validation score is retained
model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features,   # Use the updated list of categorical features
    use_best_model=True
)

## 18. Model Evaluation (Second Iteration)

The R2 score is calculated for the predictions made by the retrained model on the validation set. This evaluates if the additional features and refined hyperparameters led to an improvement in performance.

In [ ]:
# Print the column names of the training feature set (X_train) as a list
print(X_train.columns.tolist())

In [ ]:
from sklearn.metrics import r2_score

# Make predictions on the validation set using the trained model
pred = model.predict(X_valid)

# Calculate the R2 score to evaluate the model's performance
score = r2_score(y_valid, pred)

# Print the R2 score
print("R2 Score =", score)

## 19. Making Predictions on Test Data

The final trained model is used to predict traffic demand on the unseen test dataset. These predictions will be used to generate the submission file.

In [ ]:
# Use the final trained model to make predictions on the unseen test dataset
preds = model.predict(X_test)

## 20. Generating Submission File

The predictions are formatted into a pandas DataFrame along with the 'Index' from the test set. This DataFrame is then saved to a CSV file named 'submission.csv', ready for submission to a competition or for further analysis.

In [ ]:
# Create a pandas DataFrame for the submission file
# It includes the 'Index' from the original test data and the predicted 'demand'
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

# Save the submission DataFrame to a CSV file named 'submission.csv'
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
submission.to_csv('submission.csv', index=False)

# Display the first few rows of the submission DataFrame to verify its format
print(submission.head())

# Ensemble Model (CatBoost + LightGBM)

To further improve prediction performance, an ensemble approach is explored by combining predictions from CatBoost and LightGBM models.

The final prediction is obtained by averaging outputs from both models.

In [54]:
!pip install lightgbm -q

In [53]:
from lightgbm import LGBMRegressor

In [55]:
print(X.shape)
print(y.shape)
print(X_test.shape)

(77299, 16)
(77299,)
(41778, 16)


In [58]:
X_lgb = X.copy()
X_test_lgb = X_test.copy()

categorical_cols = X_lgb.select_dtypes(include=['object']).columns

for col in categorical_cols:
    X_lgb[col] = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

print("Categorical conversion completed")

Categorical conversion completed


In [59]:
lgb_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

lgb_model.fit(X_lgb, y)

print("LightGBM Training Completed")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021235 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18411
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 16
[LightGBM] [Info] Start training from score 0.093942
LightGBM Training Completed


In [64]:
# Peak Hour Feature

peak_hours = [7, 8, 9, 17, 18, 19, 20]

train['is_peak_hour'] = train['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)

test['is_peak_hour'] = test['hour'].apply(
    lambda x: 1 if x in peak_hours else 0
)

In [65]:
# Road + Weather

train['road_weather'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['Weather'].astype(str)
)

test['road_weather'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['Weather'].astype(str)
)

In [66]:
# Geohash + Hour

train['geo_hour'] = (
    train['geohash'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

test['geo_hour'] = (
    test['geohash'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

In [67]:
# Road Type + Lanes

train['road_lane'] = (
    train['RoadType'].astype(str)
    + "_"
    + train['NumberofLanes'].astype(str)
)

test['road_lane'] = (
    test['RoadType'].astype(str)
    + "_"
    + test['NumberofLanes'].astype(str)
)

In [68]:
# Weather + Hour

train['weather_hour'] = (
    train['Weather'].astype(str)
    + "_"
    + train['hour'].astype(str)
)

test['weather_hour'] = (
    test['Weather'].astype(str)
    + "_"
    + test['hour'].astype(str)
)

In [61]:
print(model)

CatBoostRegressor(depth=10, eval_metric='R2', iterations=4000, learning_rate=0.02, loss_function='RMSE', verbose=200)


In [71]:
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_weather',
    'geo_hour',
    'road_lane',
    'weather_hour'
]

In [72]:
# Preparing Features and Target

X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

In [73]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [74]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1364645	total: 160ms	remaining: 2m 39s
100:	learn: 0.0428660	total: 12.5s	remaining: 1m 50s
200:	learn: 0.0390480	total: 26.6s	remaining: 1m 45s
300:	learn: 0.0370523	total: 40.8s	remaining: 1m 34s
400:	learn: 0.0354780	total: 54.8s	remaining: 1m 21s
500:	learn: 0.0342187	total: 1m 10s	remaining: 1m 9s
600:	learn: 0.0332306	total: 1m 24s	remaining: 56.3s
700:	learn: 0.0324345	total: 1m 39s	remaining: 42.4s
800:	learn: 0.0318186	total: 1m 53s	remaining: 28.2s
900:	learn: 0.0313229	total: 2m 7s	remaining: 14s
999:	learn: 0.0308573	total: 2m 25s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [75]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.9413759738997606


In [76]:
final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

0:	learn: 0.1362059	total: 463ms	remaining: 7m 42s
100:	learn: 0.0409013	total: 14.7s	remaining: 2m 11s
200:	learn: 0.0384640	total: 29.2s	remaining: 1m 56s
300:	learn: 0.0364875	total: 44.3s	remaining: 1m 42s
400:	learn: 0.0351368	total: 59.6s	remaining: 1m 29s
500:	learn: 0.0339518	total: 1m 15s	remaining: 1m 15s
600:	learn: 0.0330400	total: 1m 32s	remaining: 1m 1s
700:	learn: 0.0322767	total: 1m 51s	remaining: 47.4s
800:	learn: 0.0315401	total: 2m 8s	remaining: 31.9s
900:	learn: 0.0309838	total: 2m 25s	remaining: 16s
999:	learn: 0.0304835	total: 2m 43s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [77]:
preds = final_model.predict(X_test)

In [78]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

submission.to_csv(
    'submission_feature_eng.csv',
    index=False
)

In [79]:
print("Validation R²:", r2)

Validation R²: 0.9413759738997606


In [80]:
eval_model

CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [81]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

submission.head()

,Index,demand
0,0,0.045011
1,1,0.018937
2,2,0.006923
3,3,0.025372
4,4,0.055852


In [82]:
submission.to_csv(
    'submission_feature_eng.csv',
    index=False
)

print("submission_feature_eng.csv created successfully")

submission_feature_eng.csv created successfully


In [83]:
from google.colab import files

files.download('submission_feature_eng.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [84]:
# ==================================
# Geohash Mean Demand Feature
# ==================================

geo_mean = train.groupby('geohash')['demand'].mean()

train['geo_demand_mean'] = train['geohash'].map(geo_mean)
test['geo_demand_mean'] = test['geohash'].map(geo_mean)

print("geo_demand_mean feature created")

geo_demand_mean feature created


In [85]:
# ==================================
# Geohash Prefix Features
# ==================================

train['geo_4'] = train['geohash'].str[:4]
test['geo_4'] = test['geohash'].str[:4]

train['geo_5'] = train['geohash'].str[:5]
test['geo_5'] = test['geohash'].str[:5]

print("geo_4 and geo_5 features created")

geo_4 and geo_5 features created


In [86]:
cat_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_weather',
    'geo_hour',
    'road_lane',
    'weather_hour',
    'geo_4',
    'geo_5'
]

In [87]:
X = train.drop(['Index', 'demand', 'timestamp'], axis=1)
y = train['demand']

X_test = test.drop(['Index', 'timestamp'], axis=1)

In [88]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

(61839, 21)
(15460, 21)


In [89]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1361750	total: 201ms	remaining: 3m 20s
100:	learn: 0.0371148	total: 17.6s	remaining: 2m 36s
200:	learn: 0.0337131	total: 37.5s	remaining: 2m 28s
300:	learn: 0.0318913	total: 55.7s	remaining: 2m 9s
400:	learn: 0.0305798	total: 1m 17s	remaining: 1m 55s
500:	learn: 0.0297179	total: 1m 36s	remaining: 1m 35s
600:	learn: 0.0290578	total: 1m 55s	remaining: 1m 16s
700:	learn: 0.0284264	total: 2m 15s	remaining: 57.8s
800:	learn: 0.0279913	total: 2m 33s	remaining: 38.2s
900:	learn: 0.0275331	total: 2m 53s	remaining: 19.1s
999:	learn: 0.0271412	total: 3m 12s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [90]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

(61839, 21)
(15460, 21)


In [91]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.9518733733395889


In [92]:
train_preds = eval_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)

print("Train R²:", train_r2)
print("Validation R²:", r2)
print("Gap:", train_r2 - r2)

Train R²: 0.9640678196506596
Validation R²: 0.9518733733395889
Gap: 0.012194446311070695


In [93]:
final_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

0:	learn: 0.1360850	total: 435ms	remaining: 7m 14s
100:	learn: 0.0366059	total: 22.3s	remaining: 3m 18s
200:	learn: 0.0333742	total: 44.2s	remaining: 2m 55s
300:	learn: 0.0315789	total: 1m 5s	remaining: 2m 32s
400:	learn: 0.0305203	total: 1m 28s	remaining: 2m 12s
500:	learn: 0.0297197	total: 1m 52s	remaining: 1m 52s
600:	learn: 0.0290675	total: 2m 15s	remaining: 1m 29s
700:	learn: 0.0285661	total: 2m 37s	remaining: 1m 7s
800:	learn: 0.0280864	total: 3m 1s	remaining: 45s
900:	learn: 0.0276928	total: 3m 24s	remaining: 22.5s
999:	learn: 0.0273025	total: 3m 47s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [94]:
preds = final_model.predict(X_test)

In [95]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

submission.to_csv(
    'submission_geo_features.csv',
    index=False
)

submission.head()

,Index,demand
0,0,0.060105
1,1,0.023608
2,2,0.016601
3,3,0.016261
4,4,0.060495


In [96]:
from google.colab import files

files.download('submission_geo_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Peak Hour Feature

Traffic demand usually increases during morning and evening rush hours.

A binary feature is created to identify peak traffic periods.

In [97]:
# ==================================
# Geohash + Time Slot Mean Demand
# ==================================

geo_ts_mean = train.groupby(
    ['geohash', 'time_slot']
)['demand'].mean()

train['geo_ts_demand_mean'] = (
    train.set_index(['geohash', 'time_slot'])
         .index.map(geo_ts_mean)
)

test['geo_ts_demand_mean'] = (
    test.set_index(['geohash', 'time_slot'])
        .index.map(geo_ts_mean)
)

print("geo_ts_demand_mean feature created")

geo_ts_demand_mean feature created


In [98]:
X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

print(X.shape)
print(X_test.shape)

(77299, 22)
(41778, 22)


In [99]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [100]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1354807	total: 343ms	remaining: 5m 43s
100:	learn: 0.0114726	total: 21.6s	remaining: 3m 12s
200:	learn: 0.0102770	total: 39.7s	remaining: 2m 37s
300:	learn: 0.0096757	total: 59.4s	remaining: 2m 17s
400:	learn: 0.0092828	total: 1m 18s	remaining: 1m 56s
500:	learn: 0.0089523	total: 1m 38s	remaining: 1m 38s
600:	learn: 0.0086267	total: 1m 57s	remaining: 1m 18s
700:	learn: 0.0083506	total: 2m 19s	remaining: 59.3s
800:	learn: 0.0081028	total: 2m 38s	remaining: 39.3s
900:	learn: 0.0078804	total: 2m 58s	remaining: 19.6s
999:	learn: 0.0076843	total: 3m 17s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [101]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.9941906747970983


In [102]:
train_preds = eval_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)

print("Train R²:", train_r2)
print("Validation R²:", r2)
print("Gap:", train_r2 - r2)

Train R²: 0.9963158245794362
Validation R²: 0.9941906747970983
Gap: 0.0021251497823379095


In [103]:
preds = final_model.predict(X_test)

In [104]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': preds
})

submission.to_csv(
    'submission_final.csv',
    index=False
)

submission.head()

,Index,demand
0,0,0.060105
1,1,0.023608
2,2,0.016601
3,3,0.016261
4,4,0.060495


In [105]:
from google.colab import files

files.download('submission_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [107]:
print(len(X.columns))
print(len(final_model.get_feature_importance()))

22
21


In [108]:
print(X.columns.tolist())

['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_slot', 'peak_hour', 'road_lane', 'geo_hour', 'weather_hour', 'temp_bin', 'is_peak_hour', 'road_weather', 'geo_demand_mean', 'geo_4', 'geo_5', 'geo_ts_demand_mean']


In [109]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': eval_model.get_feature_importance()
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance.head(20))

               Feature  Importance
21  geo_ts_demand_mean   80.076281
1                  day    4.788864
12           road_lane    3.843277
4        LargeVehicles    2.309321
2             RoadType    1.911773
10           time_slot    1.499785
17        road_weather    1.421090
8                 hour    0.868969
19               geo_4    0.735158
20               geo_5    0.550165
18     geo_demand_mean    0.475063
14        weather_hour    0.460111
13            geo_hour    0.267450
3        NumberofLanes    0.262053
0              geohash    0.234582
6          Temperature    0.127297
7              Weather    0.075571
9               minute    0.056717
15            temp_bin    0.023128
5            Landmarks    0.013344


In [110]:
road_mean = train.groupby('RoadType')['demand'].mean()

train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)

In [111]:
weather_mean = train.groupby('Weather')['demand'].mean()

train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)

In [112]:
train['geo_day'] = (
    train['geohash'].astype(str)
    + "_"
    + train['day'].astype(str)
)

test['geo_day'] = (
    test['geohash'].astype(str)
    + "_"
    + test['day'].astype(str)
)

In [113]:
cat_features.append('geo_day')

In [114]:
X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

In [115]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [116]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1355067	total: 388ms	remaining: 6m 27s
100:	learn: 0.0116534	total: 22.3s	remaining: 3m 18s
200:	learn: 0.0102247	total: 42.9s	remaining: 2m 50s
300:	learn: 0.0095203	total: 1m 4s	remaining: 2m 30s
400:	learn: 0.0091102	total: 1m 25s	remaining: 2m 7s
500:	learn: 0.0087804	total: 1m 48s	remaining: 1m 47s
600:	learn: 0.0084772	total: 2m 9s	remaining: 1m 26s
700:	learn: 0.0082465	total: 2m 30s	remaining: 1m 4s
800:	learn: 0.0080379	total: 2m 53s	remaining: 43s
900:	learn: 0.0078198	total: 3m 14s	remaining: 21.3s
999:	learn: 0.0076255	total: 3m 36s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [117]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.9946216905472668


In [118]:
print('geo_ts_demand_mean' in X.columns)

True


In [119]:
X = X.drop('geo_ts_demand_mean', axis=1)

X_test = X_test.drop('geo_ts_demand_mean', axis=1)

In [120]:
road_mean = train.groupby('RoadType')['demand'].mean()

train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)

In [121]:
weather_mean = train.groupby('Weather')['demand'].mean()

train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)

In [122]:
X = train.drop(
    ['Index','demand','timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index','timestamp'],
    axis=1
)

In [123]:
print('geo_ts_demand_mean' in X.columns)

True


In [124]:
X = X.drop('geo_ts_demand_mean', axis=1)
X_test = X_test.drop('geo_ts_demand_mean', axis=1)

In [125]:
road_mean = train.groupby('RoadType')['demand'].mean()

train['road_demand_mean'] = train['RoadType'].map(road_mean)
test['road_demand_mean'] = test['RoadType'].map(road_mean)

In [126]:
weather_mean = train.groupby('Weather')['demand'].mean()

train['weather_demand_mean'] = train['Weather'].map(weather_mean)
test['weather_demand_mean'] = test['Weather'].map(weather_mean)

In [127]:
X = train.drop(
    ['Index', 'demand', 'timestamp'],
    axis=1
)

y = train['demand']

X_test = test.drop(
    ['Index', 'timestamp'],
    axis=1
)

In [128]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [129]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1355067	total: 381ms	remaining: 6m 20s
100:	learn: 0.0116534	total: 23.2s	remaining: 3m 26s
200:	learn: 0.0102247	total: 42.8s	remaining: 2m 50s
300:	learn: 0.0095203	total: 1m 4s	remaining: 2m 30s
400:	learn: 0.0091102	total: 1m 26s	remaining: 2m 9s
500:	learn: 0.0087804	total: 1m 47s	remaining: 1m 47s
600:	learn: 0.0084772	total: 2m 9s	remaining: 1m 26s
700:	learn: 0.0082465	total: 2m 31s	remaining: 1m 4s
800:	learn: 0.0080379	total: 2m 52s	remaining: 42.9s
900:	learn: 0.0078198	total: 3m 14s	remaining: 21.4s
999:	learn: 0.0076255	total: 3m 35s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [130]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.9946216905472668


In [131]:
train_preds = eval_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)

print("Train R²:", train_r2)
print("Validation R²:", r2)
print("Gap:", train_r2 - r2)

Train R²: 0.9964926247095972
Validation R²: 0.9946216905472668
Gap: 0.0018709341623304176


In [132]:
print('geo_ts_demand_mean' in X.columns)

True


In [133]:
print(X.columns.tolist())

['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_slot', 'peak_hour', 'road_lane', 'geo_hour', 'weather_hour', 'temp_bin', 'is_peak_hour', 'road_weather', 'geo_demand_mean', 'geo_4', 'geo_5', 'geo_ts_demand_mean', 'road_demand_mean', 'weather_demand_mean', 'geo_day']


In [134]:
[col for col in X.columns if 'geo' in col]

['geohash',
 'geo_hour',
 'geo_demand_mean',
 'geo_4',
 'geo_5',
 'geo_ts_demand_mean',
 'geo_day']

In [135]:
if 'geo_ts_demand_mean' in X.columns:
    X.drop('geo_ts_demand_mean', axis=1, inplace=True)

if 'geo_ts_demand_mean' in X_test.columns:
    X_test.drop('geo_ts_demand_mean', axis=1, inplace=True)

print('geo_ts_demand_mean' in X.columns)
print('geo_ts_demand_mean' in X_test.columns)

False
False


In [136]:
print('geo_ts_demand_mean' in X.columns)
print(X.shape)

False
(77299, 24)


In [137]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [138]:
from catboost import CatBoostRegressor

eval_model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=100
)

eval_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1361481	total: 384ms	remaining: 6m 23s
100:	learn: 0.0371407	total: 25.9s	remaining: 3m 50s
200:	learn: 0.0334754	total: 47.7s	remaining: 3m 9s
300:	learn: 0.0315250	total: 1m 8s	remaining: 2m 38s
400:	learn: 0.0303584	total: 1m 31s	remaining: 2m 16s
500:	learn: 0.0295366	total: 1m 54s	remaining: 1m 53s
600:	learn: 0.0288556	total: 2m 15s	remaining: 1m 29s
700:	learn: 0.0282303	total: 2m 37s	remaining: 1m 7s
800:	learn: 0.0277603	total: 2m 59s	remaining: 44.7s
900:	learn: 0.0273267	total: 3m 21s	remaining: 22.1s
999:	learn: 0.0269086	total: 3m 42s	remaining: 0us


CatBoostRegressor(depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [139]:
from sklearn.metrics import r2_score

val_preds = eval_model.predict(X_val)

r2 = r2_score(y_val, val_preds)

print("Validation R²:", r2)

Validation R²: 0.952810109371291
